 # Fraud Detection - Dimensional Warehouse & OLAP

 **Analytical question:** Where and when is fraud most likely, and where does it cost the most money?

 **Fact grain:** one row = one transaction

 **Measures:** TransactionAmt, isFraud, fraudAmt

 **Dimensions:**
 - dimProduct (ProductCD)
 - dimCard    (card6)
 - dimDevice  (DeviceType)
 - dimTime    (Month -> Day -> Hour)

In [ ]:
import os
import pandas as pd

In [ ]:
df = pd.read_csv(
    "../Data/train_transaction.csv",
    usecols=["TransactionID", "isFraud", "TransactionDT",
            "TransactionAmt", "ProductCD", "card6"],
)
print("Raw shape:", df.shape)

Raw shape: (590540, 6)


In [ ]:
ident = pd.read_csv("../Data/train_identity.csv",
                    usecols=["TransactionID", "DeviceType"])
assert ident["TransactionID"].is_unique, "Identity has duplicate TransactionIDs!"

df = df.merge(ident, on="TransactionID", how="left", validate="one_to_one")
print("Shape after identity join:", df.shape)

Shape after identity join: (590540, 7)


In [ ]:
# Missing values in dimension attributes -> explicit "Unknown" label
df["card6"] = df["card6"].fillna("Unknown")
df["DeviceType"] = df["DeviceType"].fillna("Unknown")   # no identity record

# TransactionDT = seconds from a start point
df["Day"] = df["TransactionDT"] // 86400
df["Month"] = df["Day"] // 30 + 1
df["Hour"] = (df["TransactionDT"] // 3600) % 24

# new measure: money lost to fraud (0 for normal transactions)
df["fraudAmt"] = df["TransactionAmt"] * df["isFraud"]

In [ ]:
dimProduct = df[["ProductCD"]].drop_duplicates().reset_index(drop=True)
dimProduct["ProductKey"] = range(1, len(dimProduct) + 1)

dimCard = df[["card6"]].drop_duplicates().reset_index(drop=True)
dimCard["CardKey"] = range(1, len(dimCard) + 1)

dimDevice = df[["DeviceType"]].drop_duplicates().reset_index(drop=True)
dimDevice["DeviceKey"] = range(1, len(dimDevice) + 1)

dimTime = df[["Month", "Day", "Hour"]].drop_duplicates().reset_index(drop=True)
dimTime["TimeKey"] = range(1, len(dimTime) + 1)

In [ ]:
rowsBefore = len(df)

fact = (
    df.merge(dimProduct, on="ProductCD", how="left", validate="many_to_one")
        .merge(dimCard, on="card6", how="left", validate="many_to_one")
        .merge(dimDevice, on="DeviceType", how="left", validate="many_to_one")
        .merge(dimTime[["Day", "Hour", "TimeKey"]], on=["Day", "Hour"], how="left", validate="many_to_one")
        [["TransactionID", "ProductKey", "CardKey", "DeviceKey", "TimeKey", "TransactionAmt", "isFraud", "fraudAmt"]]
)

In [ ]:
print("Rows before:", rowsBefore, "| after:", len(fact))
assert len(fact) == rowsBefore, "Fan-out detected - fix the join!"
assert fact["TransactionID"].is_unique, "Duplicate transactions!"
assert fact[["ProductKey", "CardKey", "DeviceKey", "TimeKey"]].notna().all().all(), "Missing keys!"
print("PASS: grain is still one row per transaction")

Rows before: 590540 | after: 590540
PASS: grain is still one row per transaction


In [ ]:
cube = (
    fact.merge(dimProduct, on="ProductKey")
        .merge(dimCard, on="CardKey")
        .merge(dimDevice, on="DeviceKey")
        .merge(dimTime, on="TimeKey")
)

overallRate = cube["isFraud"].mean()
print(f"Overall fraud rate: {overallRate:.2%}")

Overall fraud rate: 3.50%


In [ ]:
def summary(data, by):
    s = data.groupby(by).agg(
        transactions=("TransactionID", "count"),
        fraudCases=("isFraud", "sum"),
        fraudRate=("isFraud", "mean"),
        fraudAmt=("fraudAmt", "sum"),
    )
    s["lift"] = s["fraudRate"] / overallRate   # 2.0 = twice as risky as average
    s["share_of_fraud_$"] = s["fraudAmt"] / cube["fraudAmt"].sum()
    return s.round(3)

In [ ]:
print(summary(cube, "ProductCD").sort_values("lift", ascending=False))

           transactions  fraudCases  fraudRate    fraudAmt   lift  \
ProductCD                                                           
C                 68519        8008      0.117   391421.40  3.340   
S                 11628         686      0.059    43416.00  1.686   
H                 33024        1574      0.048   246632.00  1.362   
R                 37699        1426      0.038   348050.00  1.081   
W                439670        8969      0.020  2054325.46  0.583   

           share_of_fraud_$  
ProductCD                    
C                     0.127  
S                     0.014  
H                     0.080  
R                     0.113  
W                     0.666  


In [ ]:
print(summary(cube, "card6").sort_values("lift", ascending=False))

                 transactions  fraudCases  fraudRate     fraudAmt   lift  \
card6                                                                      
credit                 148986        9950      0.067  1656751.371  1.909   
Unknown                  1571          39      0.025     5051.209  0.709   
debit                  439938       10674      0.024  1422042.280  0.693   
charge card                15           0      0.000        0.000  0.000   
debit or credit            30           0      0.000        0.000  0.000   

                 share_of_fraud_$  
card6                              
credit                      0.537  
Unknown                     0.002  
debit                       0.461  
charge card                 0.000  
debit or credit             0.000  


In [ ]:
print(summary(cube, "DeviceType").sort_values("lift", ascending=False))

            transactions  fraudCases  fraudRate     fraudAmt   lift  \
DeviceType                                                            
mobile             55645        5657      0.102   546427.465  2.905   
desktop            85165        5554      0.065   449878.937  1.864   
Unknown           449730        9452      0.021  2087538.458  0.601   

            share_of_fraud_$  
DeviceType                    
mobile                 0.177  
desktop                0.146  
Unknown                0.677  


In [ ]:
print(summary(cube, "Month"))

       transactions  fraudCases  fraudRate    fraudAmt   lift  \
Month                                                           
1            130968        3243      0.025  439589.886  0.708   
2             89838        3627      0.040  496493.411  1.154   
3             91768        3700      0.040  515459.252  1.152   
4             98027        3849      0.039  643237.323  1.122   
5             85303        2962      0.035  493820.591  0.992   
6             86525        2943      0.034  435285.633  0.972   
7              8111         339      0.042   59958.764  1.194   

       share_of_fraud_$  
Month                    
1                 0.143  
2                 0.161  
3                 0.167  
4                 0.209  
5                 0.160  
6                 0.141  
7                 0.019  


In [ ]:
print(summary(cube[cube["Month"] == 1], "Day"))

     transactions  fraudCases  fraudRate   fraudAmt   lift  share_of_fraud_$
Day                                                                         
1            5122         112      0.022  20337.144  0.625             0.007
2            3730         123      0.033  15359.607  0.942             0.005
3            3241          92      0.028  14254.337  0.811             0.005
4            4036         115      0.028  14120.447  0.814             0.005
5            3964         127      0.032  16692.721  0.916             0.005
6            3717          99      0.027  12869.080  0.761             0.004
7            3786         136      0.036  21670.363  1.027             0.007
8            3999          93      0.023  13508.152  0.665             0.004
9            3907         119      0.030  13106.598  0.870             0.004
10           3445          90      0.026  12709.721  0.747             0.004
11           4319         123      0.028  21725.539  0.814             0.007

In [ ]:
print(summary(cube, "Hour"))

      transactions  fraudCases  fraudRate    fraudAmt   lift  share_of_fraud_$
Hour                                                                          
0            37795        1186      0.031  167359.236  0.897             0.054
1            32797        1027      0.031  139679.913  0.895             0.045
2            26732        1002      0.037  122812.961  1.071             0.040
3            20802         797      0.038   93367.785  1.095             0.030
4            14839         770      0.052   83123.128  1.483             0.027
5             9701         682      0.070   79136.279  2.009             0.026
6             6007         467      0.078   50704.911  2.222             0.016
7             3704         393      0.106   39466.933  3.032             0.013
8             2591         241      0.093   38833.631  2.658             0.013
9             2479         223      0.090   48447.958  2.571             0.016
10            3627         193      0.053   31333.13

In [ ]:
print(summary(cube[cube["ProductCD"] == "C"], "Month"))

       transactions  fraudCases  fraudRate   fraudAmt   lift  share_of_fraud_$
Month                                                                         
1             15828        1262      0.080  72739.746  2.279             0.024
2             10105        1331      0.132  59829.361  3.764             0.019
3             11048        1345      0.122  52791.202  3.479             0.017
4             12684        1600      0.126  77673.533  3.605             0.025
5              8874        1123      0.127  61970.591  3.617             0.020
6              9294        1225      0.132  60607.353  3.767             0.020
7               686         122      0.178   5809.614  5.083             0.002


In [ ]:
dice = cube[
    cube["ProductCD"].isin(["W", "C"])
    & (cube["card6"] == "credit")
    & cube["Month"].isin([1, 2, 3])
]
print(summary(dice, ["ProductCD", "Month"]))

                 transactions  fraudCases  fraudRate    fraudAmt   lift  \
ProductCD Month                                                           
C         1              6122         775      0.127   46040.547  3.618   
          2              4329         773      0.179   34197.294  5.103   
          3              4662         896      0.192   35864.869  5.493   
W         1             12785         456      0.036  105048.480  1.019   
          2             12771         569      0.045  133950.350  1.273   
          3             11670         581      0.050  158648.320  1.423   

                 share_of_fraud_$  
ProductCD Month                    
C         1                 0.015  
          2                 0.011  
          3                 0.012  
W         1                 0.034  
          2                 0.043  
          3                 0.051  


In [ ]:
pivot = cube.pivot_table(
    index="ProductCD", columns="Month", values="isFraud", aggfunc="mean"
)
print(pivot.round(3))

Month          1      2      3      4      5      6      7
ProductCD                                                 
C          0.080  0.132  0.122  0.126  0.127  0.132  0.178
H          0.018  0.134  0.128  0.099  0.059  0.063  0.061
R          0.010  0.095  0.091  0.062  0.050  0.047  0.079
S          0.025  0.071  0.107  0.079  0.064  0.047  0.157
W          0.019  0.021  0.021  0.022  0.021  0.018  0.022


In [ ]:
print(cube.pivot_table(
    index="ProductCD", columns="card6", values="isFraud", aggfunc="mean"
).round(3))

card6      Unknown  charge card  credit  debit  debit or credit
ProductCD                                                      
C            0.093          0.0   0.169  0.082              NaN
H            0.000          NaN   0.050  0.045              NaN
R            0.000          0.0   0.039  0.035              NaN
S            0.000          NaN   0.057  0.062              NaN
W            0.015          NaN   0.042  0.016              0.0


In [ ]:
print(cube.pivot_table(
    index="ProductCD", columns="DeviceType", values="isFraud", aggfunc="mean"
).round(3))

DeviceType  Unknown  desktop  mobile
ProductCD                           
C             0.055    0.118   0.131
H             0.012    0.043   0.056
R             0.017    0.022   0.075
S             0.103    0.044   0.116
W             0.020      NaN     NaN


In [ ]:
tablesPath = "../Tables/"
os.makedirs(tablesPath, exist_ok=True)

cube.to_csv(os.path.join(tablesPath, "fraud_cube.csv"), index=False)
fact.to_csv(os.path.join(tablesPath, "fact_transactions.csv"), index=False)
dimProduct.to_csv(os.path.join(tablesPath, "dimProduct.csv"), index=False)
dimCard.to_csv(os.path.join(tablesPath, "dimCard.csv"), index=False)
dimDevice.to_csv(os.path.join(tablesPath, "dimDevice.csv"), index=False)
dimTime.to_csv(os.path.join(tablesPath, "dimTime.csv"), index=False)